# Addis Ababa Ride Demand Forecasting
## 04 — Modeling & Evaluation (Deliverable D)

* Input: `data/processed/master_train.csv`, `master_test.csv` from notebook 01.
* **Main split (Rule 7):** train on everything **before 18 Oct 2025 00:00**, validate on **18–31 Oct 2025** (14 days = the forecast horizon).
* **Rolling origin:** 4 folds with cut dates 6 Sep, 20 Sep, 4 Oct, 18 Oct; each validates on the following 14 days.
* Nothing is fitted on the test file (Rule 8). Outputs: `models/final_model.joblib`, `submission/team_alpha_submission.csv`,
  and `reports/model_results.csv`, `val_predictions.csv`, `feature_importance.csv` (used by notebook 03 for fig10–12).

In [30]:
from pathlib import Path
import time, warnings, itertools
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
PROC, REP, MOD, SUB = ROOT / "data" / "processed", ROOT / "reports", ROOT / "models", ROOT / "submission"
for d in (REP, MOD, SUB): d.mkdir(exist_ok=True)

train = pd.read_csv(PROC / "master_train.csv", parse_dates=["datetime"])
test = pd.read_csv(PROC / "master_test.csv", parse_dates=["datetime"])
events = pd.read_csv(PROC / "events_processed.csv", parse_dates=["start", "end"])
ZONES = sorted(train.zone.unique())

# rain_class -> ordinal so every model can use it
RAIN_ORD = {"none": 0, "light": 1, "moderate": 2, "heavy": 3}
for df in (train, test):
    df["rain_class_ord"] = df["rain_class"].map(RAIN_ORD)
    df["level"] = df["roll_mean_168h_lag336"]          # zone's recent level (known 14 days ahead)

print("train", train.shape, train.datetime.min(), "->", train.datetime.max())
print("test ", test.shape, test.datetime.min(), "->", test.datetime.max())

train (82979, 39) 2025-01-01 00:00:00 -> 2025-10-31 23:00:00
test  (4032, 36) 2025-11-01 00:00:00 -> 2025-11-14 23:00:00


## D4 — Feature availability & leakage audit

In [31]:
CALENDAR = ["zone", "hour", "day_of_week", "is_weekend", "day_of_month", "is_payday_window",
            "is_public_holiday", "is_school_break", "days_since_start", "hour_sin", "hour_cos"]
LAGS = ["lag_336h", "lag_504h", "roll_mean_168h_lag336"]
WEATHER = ["temp_c", "rain_mm", "humidity_pct", "wind_kmh", "rain_3h_sum", "is_raining", "rain_class_ord"]
EVENTS = ["event_in_window", "event_attendance", "football_window", "concert_window", "conference_window",
          "road_closure_window", "event_pre", "event_during", "event_post"]
FEATS = CALENDAR + LAGS + WEATHER + EVENTS

audit = []
for f in CALENDAR: audit.append((f, "calendar", "yes", "derived from the date/zone"))
for f in LAGS: audit.append((f, "trip history", "yes", "lag ≥ 336 h = forecast horizon, so the value exists on forecast day"))
for f in WEATHER: audit.append((f, "weather", "yes (as forecast)", "weather forecast rows exist for 1–14 Nov; noisier than observed"))
for f in EVENTS: audit.append((f, "events", "yes", "events are scheduled in advance"))
for f, why in [("month", "Nov never seen in training – trees cannot extrapolate; trend handled by days_since_start/lags"),
               ("lag_1h / lag_24h", "not known 2–14 days ahead (removed in notebook 01)"),
               ("active_drivers", "consequence of demand, only exists in history"),
               ("avg_wait_min", "consequence of demand, only exists in history"),
               ("avg_fare_birr", "only exists in history"),
               ("record_id / row_id / pickup_hour / datetime", "identifiers")]:
    audit.append((f, "excluded", "no / not used", why))
audit = pd.DataFrame(audit, columns=["feature", "source", "known_at_forecast_time", "note"])
display(audit)
assert set(FEATS) <= set(test.columns), "every model feature must exist in the test table"
print(len(FEATS), "features used")

,feature,source,known_at_forecast_time,note
0,zone,calendar,yes,derived from the date/zone
1,hour,calendar,yes,derived from the date/zone
2,day_of_week,calendar,yes,derived from the date/zone
3,is_weekend,calendar,yes,derived from the date/zone
4,day_of_month,calendar,yes,derived from the date/zone
5,is_payday_window,calendar,yes,derived from the date/zone
6,is_public_holiday,calendar,yes,derived from the date/zone
7,is_school_break,calendar,yes,derived from the date/zone
8,days_since_start,calendar,yes,derived from the date/zone
9,hour_sin,calendar,yes,derived from the date/zone


30 features used


## Helpers: split, models, metrics

In [32]:
VAL_START = pd.Timestamp("2025-10-18")
def split(df, cut, days=14):
    return df[df.datetime < cut], df[(df.datetime >= cut) & (df.datetime < cut + pd.Timedelta(days=days))]
tr_main, va_main = split(train, VAL_START)
print(f"train: {tr_main.datetime.min():%d %b} – {tr_main.datetime.max():%d %b %H:%M} ({len(tr_main):,} rows)")
print(f"valid: {va_main.datetime.min():%d %b} – {va_main.datetime.max():%d %b %H:%M} ({len(va_main):,} rows)")

rmse = lambda y, p: float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))
mae = lambda y, p: float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))

def X_of(df, feats):
    X = df[feats].copy()
    if "zone" in X: X["zone"] = pd.Categorical(X["zone"], categories=ZONES)
    return X

BASE_LGB = dict(n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=20,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=0.0, random_state=SEED, verbose=-1, n_jobs=-1)

def sklearn_pipe(kind, feats):
    cat = [c for c in ["zone", "hour", "day_of_week"] if c in feats]
    num = [c for c in feats if c not in cat]
    num_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()) if kind == "ridge" \
               else SimpleImputer(strategy="median")
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat), ("num", num_pipe, num)])
    est = Ridge(alpha=10.0, random_state=SEED) if kind == "ridge" else \
          RandomForestRegressor(n_estimators=150, max_depth=18, min_samples_leaf=3, n_jobs=-1, random_state=SEED)
    return make_pipeline(pre, est)

def fit(kind, tr, feats, params=None):
    """kind: 'lgb' (raw target), 'lgb_norm' (target / zone level, removes the growth trend), 'ridge', 'rf'."""
    if kind in ("lgb", "lgb_norm"):
        mdl = lgb.LGBMRegressor(**{**BASE_LGB, **(params or {})})
        if kind == "lgb":
            mdl.fit(X_of(tr, feats), tr.trips)
        else:
            t2 = tr[tr.level.notna()]
            mdl.fit(X_of(t2, feats), t2.trips / t2.level.clip(lower=1))
    else:
        mdl = sklearn_pipe(kind, feats).fit(tr[feats], tr.trips)
    return {"kind": kind, "model": mdl, "features": feats}

def predict(fm, df):
    k, mdl, feats = fm["kind"], fm["model"], fm["features"]
    if k == "lgb":       p = mdl.predict(X_of(df, feats))
    elif k == "lgb_norm":
        lvl = df.level.fillna(df.lag_336h).fillna(df.lag_504h)   # fallback if a level is missing
        p = mdl.predict(X_of(df, feats)) * lvl.clip(lower=1)
        p = np.where(np.isnan(p), 0, p)
    else:                p = mdl.predict(df[feats])
    return np.clip(p, 0, None)

train: 01 Jan – 17 Oct 23:00 (79,070 rows)
valid: 18 Oct – 31 Oct 23:00 (3,909 rows)


## D1 — Baselines

In [33]:
def baseline_mean(tr, va): return np.full(len(va), tr.trips.mean())
def baseline_seasonal(tr, va, weeks=None):
    src = tr if weeks is None else tr[tr.datetime >= tr.datetime.max() - pd.Timedelta(weeks=weeks)]
    look = src.groupby(["zone", "day_of_week", "hour"])["trips"].mean()
    p = va.set_index(["zone", "day_of_week", "hour"]).index.map(look)
    return np.nan_to_num(np.asarray(p, dtype=float), nan=tr.trips.mean())

rows = []
for name, p in [("Mean predictor", baseline_mean(tr_main, va_main)),
                ("Seasonal-naive (all history)", baseline_seasonal(tr_main, va_main)),
                ("Seasonal-naive (last 4 weeks)", baseline_seasonal(tr_main, va_main, weeks=4))]:
    rows.append({"model": name, "RMSE": rmse(va_main.trips, p), "MAE": mae(va_main.trips, p)})
base_tbl = pd.DataFrame(rows).round(3)
display(base_tbl)

,model,RMSE,MAE
0,Mean predictor,27.535,20.688
1,Seasonal-naive (all history),11.301,7.359
2,Seasonal-naive (last 4 weeks),9.943,6.638


**Interpretation:** The mean predictor is useless (RMSE 27.5). The seasonal-naive baseline (mean of the same zone × weekday × hour) gets RMSE 11.3 / MAE 7.4 using all history, and **9.9 / 6.6 using only the last 4 weeks** — the recent version wins because demand grew ~40% over the year, so old weeks drag the average down. Any real model must beat 9.94.

## D2 — Model comparison (same split, same features)

In [34]:
results, preds = [], {}
for name, kind in [("Ridge regression", "ridge"), ("Random forest", "rf"),
                   ("LightGBM (raw target)", "lgb"), ("LightGBM (level-normalised target)", "lgb_norm")]:
    t0 = time.time()
    fm = fit(kind, tr_main, FEATS)
    secs = time.time() - t0
    p = predict(fm, va_main); preds[name] = p
    results.append({"model": name, "kind": kind, "RMSE": rmse(va_main.trips, p), "MAE": mae(va_main.trips, p),
                    "train_time_s": round(secs, 1)})
cmp_tbl = pd.concat([base_tbl.assign(kind="baseline", train_time_s=0.0), pd.DataFrame(results)], ignore_index=True)
display(cmp_tbl.sort_values("RMSE").round(3))
best = pd.DataFrame(results).sort_values("RMSE").iloc[0]
FINAL_KIND = best.kind
print("Winner:", best.model, "| RMSE", round(best.RMSE, 3))

,model,RMSE,MAE,kind,train_time_s
5,LightGBM (raw target),9.088,6.084,lgb,19.9
6,LightGBM (level-normalised target),9.209,6.110,lgb_norm,2.3
2,Seasonal-naive (last 4 weeks),9.943,6.638,baseline,0.0
4,Random forest,10.306,6.758,rf,29.9
3,Ridge regression,11.295,7.545,ridge,0.5
1,Seasonal-naive (all history),11.301,7.359,baseline,0.0
0,Mean predictor,27.535,20.688,baseline,0.0


Winner: LightGBM (raw target) | RMSE 9.088


**Interpretation:** LightGBM with the raw target wins (RMSE 9.09, MAE 6.08, 2 s to train), ahead of the level-normalised LightGBM (9.21), random forest (10.31) and ridge (11.30). Gradient boosting handles the zone × hour × weekday interactions and the non-linear rain/event effects natively, and the 14-day lags give it the recent level so it copes with the trend; ridge cannot model interactions without hand-built terms. The winner beats the strongest baseline by ~9% RMSE.

## D6 — Hyperparameter tuning (random search, time-ordered folds)

In [35]:
# Tuned on the three EARLIER rolling folds (cuts 6 Sep, 20 Sep, 4 Oct) so the main validation fortnight stays untouched.
TUNE_CUTS = [pd.Timestamp(d) for d in ["2025-09-06", "2025-09-20", "2025-10-04"]]
space = {"num_leaves": [15, 31, 63, 127], "learning_rate": [0.02, 0.03, 0.05, 0.08],
         "n_estimators": [300, 600, 1000], "min_child_samples": [10, 20, 50, 100],
         "colsample_bytree": [0.6, 0.8, 1.0], "reg_lambda": [0.0, 1.0, 5.0]}
rng = np.random.default_rng(SEED)
N_TRIALS = 20

def cv_score(params):
    s = []
    for cut in TUNE_CUTS:
        tr, va = split(train, cut)
        s.append(rmse(va.trips, predict(fit(FINAL_KIND, tr, FEATS, params), va)))
    return float(np.mean(s))

trials = [{"trial": 0, **{k: BASE_LGB[k] for k in space}, "cv_rmse": cv_score({})}]
for i in range(1, N_TRIALS + 1):
    params = {k: v[rng.integers(len(v))] for k, v in space.items()}
    trials.append({"trial": i, **params, "cv_rmse": cv_score(params)})
trials = pd.DataFrame(trials).sort_values("cv_rmse")
display(trials.head(8).round(4))
BEST_PARAMS = {k: (int(v) if k in ("num_leaves", "n_estimators", "min_child_samples") else float(v))
               for k, v in trials.iloc[0][list(space)].items()}
print("Search space:", space); print("Trials:", N_TRIALS, "+ default"); print("Best params:", BEST_PARAMS)

before = rmse(va_main.trips, predict(fit(FINAL_KIND, tr_main, FEATS), va_main))
after = rmse(va_main.trips, predict(fit(FINAL_KIND, tr_main, FEATS, BEST_PARAMS), va_main))
print(f"Main validation RMSE  default {before:.3f}  ->  tuned {after:.3f}")
if after > before:
    print("Tuned params do not beat defaults on the main fortnight; keeping defaults.")
    BEST_PARAMS = {}

,trial,num_leaves,learning_rate,n_estimators,min_child_samples,colsample_bytree,reg_lambda,cv_rmse
14,14,63,0.05,1000,20,0.6,5.0,9.8741
16,16,31,0.08,1000,20,1.0,0.0,9.8792
0,0,63,0.05,600,20,0.8,0.0,9.9882
9,9,63,0.08,1000,10,0.8,1.0,10.0320
4,4,127,0.03,600,20,0.6,5.0,10.0751
3,3,63,0.08,1000,100,0.8,0.0,10.1405
15,15,63,0.05,300,20,1.0,5.0,10.1652
7,7,127,0.08,300,50,0.6,5.0,10.1887


Search space: {'num_leaves': [15, 31, 63, 127], 'learning_rate': [0.02, 0.03, 0.05, 0.08], 'n_estimators': [300, 600, 1000], 'min_child_samples': [10, 20, 50, 100], 'colsample_bytree': [0.6, 0.8, 1.0], 'reg_lambda': [0.0, 1.0, 5.0]}
Trials: 20 + default
Best params: {'num_leaves': 63, 'learning_rate': 0.05, 'n_estimators': 1000, 'min_child_samples': 20, 'colsample_bytree': 0.6, 'reg_lambda': 5.0}
Main validation RMSE  default 9.088  ->  tuned 9.058


**Interpretation:** A 20-trial random search over 6 LightGBM hyperparameters, scored by mean RMSE on the three earlier rolling folds (the main fortnight was not used for tuning), picked num_leaves 63, learning_rate 0.05, 1000 trees, min_child_samples 20, colsample 0.6, reg_lambda 5. Fold CV RMSE improved from 9.99 to 9.87, and the main validation RMSE from 9.088 to 9.058 — a small gain, as is typical once features are good.

## D3 — Rolling-origin validation

In [36]:
CUTS = TUNE_CUTS + [VAL_START]
roll = []
for cut in CUTS:
    tr, va = split(train, cut)
    fm = fit(FINAL_KIND, tr, FEATS, BEST_PARAMS)
    p_m, p_s = predict(fm, va), baseline_seasonal(tr, va)
    roll.append({"cut": cut.date(), "val_window": f"{cut:%d %b} – {(cut + pd.Timedelta(days=13)):%d %b}",
                 "final_RMSE": rmse(va.trips, p_m), "final_MAE": mae(va.trips, p_m),
                 "seasonal_RMSE": rmse(va.trips, p_s), "seasonal_MAE": mae(va.trips, p_s),
                 "holidays_in_window": int(va.groupby("datetime").is_public_holiday.first().sum() / 24)})
roll = pd.DataFrame(roll)
display(roll.round(3))
summ = roll[["final_RMSE", "seasonal_RMSE"]].agg(["mean", "std"]).round(3)
display(summ)

,cut,val_window,final_RMSE,final_MAE,seasonal_RMSE,seasonal_MAE,holidays_in_window
0,2025-09-06,06 Sep – 19 Sep,9.763,6.255,13.448,8.035,1
1,2025-09-20,20 Sep – 03 Oct,10.813,6.660,14.231,8.477,1
2,2025-10-04,04 Oct – 17 Oct,9.046,5.993,10.303,6.658,0
3,2025-10-18,18 Oct – 31 Oct,9.058,6.073,11.301,7.359,0


,final_RMSE,seasonal_RMSE
mean,9.670,12.321
std,0.832,1.829


**Interpretation:** Across 4 folds the final model scores **RMSE 9.67 ± 0.83** vs **12.32 ± 1.83** for seasonal-naive (all history): it is better in every fold and roughly half as variable. The worst fold is 20 Sep – 3 Oct (10.81), which contains Meskel (27 Sep) and its eve — a holiday whose effect is local (Kazanchis/Piassa up, business zones down) and seen only once in training, plus the unlisted Meskel-eve spike found in B3.4. The two October folds are the easiest (~9.05).

## D4 (cont.) — What a leaky feature would do

In [37]:
leak_feats = FEATS + ["active_drivers"]
p_leak = predict(fit(FINAL_KIND, tr_main, leak_feats, BEST_PARAMS), va_main)
p_ok = predict(fit(FINAL_KIND, tr_main, FEATS, BEST_PARAMS), va_main)
print(f"Validation RMSE with active_drivers (LEAKY): {rmse(va_main.trips, p_leak):.3f}")
print(f"Validation RMSE without (honest):            {rmse(va_main.trips, p_ok):.3f}")
print("active_drivers is not in master_test – the leaky model could not even be run in production.")

Validation RMSE with active_drivers (LEAKY): 4.245
Validation RMSE without (honest):            9.058
active_drivers is not in master_test – the leaky model could not even be run in production.


**Interpretation:** Every model feature is known at forecast time: calendar fields, 14/21-day lags, the weather *forecast* and the scheduled event calendar. Excluded: active_drivers, avg_wait_min, avg_fare_birr (consequences of demand that only exist in history), 1 h/24 h lags (unknown 2–14 days ahead) and month (November is unseen). Adding active_drivers cuts validation RMSE from 9.06 to **4.25** — a fake 53% improvement, because the driver count is effectively a measurement of the demand being predicted; it does not exist for November, so the model could not even run in production.

## D5 — Ablation

In [38]:
sets = {"(i) calendar + zone + trend": CALENDAR + LAGS,
        "(ii) + weather": CALENDAR + LAGS + WEATHER,
        "(iii) + events": CALENDAR + LAGS + EVENTS,
        "(iv) + weather + events (full)": FEATS}
abl = []
for name, fs in sets.items():
    p = predict(fit(FINAL_KIND, tr_main, fs, BEST_PARAMS), va_main)
    abl.append({"feature_set": name, "n_features": len(fs), "RMSE": rmse(va_main.trips, p), "MAE": mae(va_main.trips, p)})
abl = pd.DataFrame(abl)
abl["ΔRMSE vs (i)"] = abl.RMSE - abl.RMSE.iloc[0]
abl["ΔRMSE %"] = 100 * abl["ΔRMSE vs (i)"] / abl.RMSE.iloc[0]
display(abl.round(3))

,feature_set,n_features,RMSE,MAE,ΔRMSE vs (i),ΔRMSE %
0,(i) calendar + zone + trend,14,9.744,6.384,0.000,0.000
1,(ii) + weather,21,9.214,6.193,-0.531,-5.446
2,(iii) + events,23,9.637,6.312,-0.108,-1.104
3,(iv) + weather + events (full),30,9.058,6.073,-0.687,-7.048


**Interpretation:** Calendar + zone + trend/lag features alone give RMSE 9.74. Adding weather cuts it to 9.21 (−5.4%), events alone to 9.64 (−1.1%), and both together to **9.06 (−7.0%)** — the gains are roughly additive. Both joins were worth it: weather is the bigger win because rain hits every zone for many hours, while events affect few zone-hours (but those hours have large errors, see D7).

## D7 — Error analysis

In [39]:
final_val = fit(FINAL_KIND, tr_main, FEATS, BEST_PARAMS)
va = va_main.copy()
va["pred"] = predict(final_val, va)
va["err"] = va.pred - va.trips
va["abs_err"] = va.err.abs()
va["day_type"] = np.select([va.is_public_holiday == 1, va.is_weekend == 1], ["holiday", "weekend"], "weekday")

def err_tbl(by):
    g = va.groupby(by).agg(mean_trips=("trips", "mean"), RMSE=("err", lambda e: np.sqrt((e ** 2).mean())),
                           MAE=("abs_err", "mean"), bias=("err", "mean"), n=("err", "size"))
    g["MAE_%_of_mean"] = 100 * g.MAE / g.mean_trips
    return g.round(2)
print("(a) by zone"); display(err_tbl("zone").sort_values("RMSE", ascending=False))
print("(a) by hour of day"); display(err_tbl("hour").T)
print("(a) by day type"); display(err_tbl("day_type"))

(a) by zone


,mean_trips,RMSE,MAE,bias,n,MAE_%_of_mean
zone,,,,,,
Merkato,48.71,12.91,8.82,-0.27,327,18.11
Kazanchis,38.85,11.63,7.57,-0.21,330,19.49
Bole,45.79,11.55,7.94,-3.18,320,17.34
Megenagna,44.23,11.28,7.72,-0.53,324,17.45
Piassa,33.86,9.23,6.07,-1.95,323,17.91
Lideta,33.62,8.14,5.62,-1.01,322,16.73
CMC,30.02,7.49,5.46,-0.27,324,18.19
Gerji,28.83,6.88,5.14,-0.52,323,17.83
Arat Kilo,24.20,6.77,4.74,-0.13,326,19.57


(a) by hour of day


hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
mean_trips,9.73,7.65,6.25,6.68,9.48,18.12,34.29,54.60,52.97,41.67,35.70,34.66,40.49,43.89,42.19,39.22,44.75,54.07,62.81,53.49,44.62,30.35,20.83,13.21
RMSE,6.60,3.71,2.51,2.65,4.13,5.65,8.37,12.18,10.86,9.41,7.74,7.50,9.86,10.91,9.23,8.79,9.66,14.97,15.26,10.82,10.17,7.90,7.77,5.85
MAE,3.04,2.56,1.99,2.14,3.15,4.07,6.29,8.78,8.37,7.19,5.77,5.79,7.00,7.57,6.85,6.56,7.38,10.43,10.87,8.26,7.44,5.75,4.78,3.59
bias,-1.02,-0.43,0.48,0.16,-0.03,0.17,1.11,-2.61,-0.98,-0.46,-1.09,0.17,-1.09,-0.88,-1.50,-1.81,-1.97,-0.33,-2.22,-0.04,-2.47,-1.97,-1.83,-1.17
n,163.00,161.00,162.00,166.00,161.00,161.00,165.00,164.00,164.00,164.00,163.00,160.00,162.00,162.00,164.00,165.00,160.00,159.00,166.00,164.00,165.00,163.00,164.00,161.00
MAE_%_of_mean,31.29,33.52,31.89,32.02,33.25,22.47,18.33,16.08,15.80,17.25,16.16,16.70,17.30,17.26,16.24,16.74,16.49,19.29,17.30,15.44,16.67,18.94,22.96,27.16


(a) by day type


,mean_trips,RMSE,MAE,bias,n,MAE_%_of_mean
day_type,,,,,,
weekday,35.09,8.89,6.10,-1.09,2803,17.38
weekend,29.29,9.46,6.01,-0.45,1106,20.51


In [40]:
top = va.nlargest(10, "abs_err")[["zone", "datetime", "trips", "pred", "err", "rain_mm", "event_in_window",
                                  "football_window", "concert_window", "event_post", "is_weekend"]]
top["weekday"] = top.datetime.dt.day_name()
print("(b) 10 largest-error zone-hours"); display(top.round(1))
# which listed events touch these zone-days?
for _, r in top.iterrows():
    hit = events[(events.zones.fillna("").str.contains(r.zone)) &
                 (events.start.dt.normalize() <= r.datetime.normalize()) & (events.end.dt.normalize() >= r.datetime.normalize())]
    print(f"{r.zone:<10} {r.datetime:%a %d %b %H:%M}  err {r.err:+6.1f}  events that day: "
          f"{', '.join(hit.event_name + ' [' + hit.status + ']') if len(hit) else '—'}")

(b) 10 largest-error zone-hours


,zone,datetime,trips,pred,err,rain_mm,event_in_window,football_window,concert_window,event_post,is_weekend,weekday
79270,Kazanchis,2025-10-18 17:00:00,36.0,107.2,71.2,0.0,1.0,1.0,0.0,1.0,1,Saturday
79326,Bole,2025-10-18 22:00:00,137.0,76.6,-60.4,0.0,1.0,0.0,0.0,0.0,1,Saturday
81518,Bole,2025-10-26 18:00:00,137.0,78.6,-58.4,0.0,1.0,0.0,1.0,0.0,1,Sunday
82077,Megenagna,2025-10-28 18:00:00,174.0,115.9,-58.1,6.5,0.0,0.0,0.0,0.0,0,Tuesday
79159,Merkato,2025-10-18 07:00:00,137.0,79.7,-57.3,0.0,0.0,0.0,0.0,0.0,1,Saturday
79229,Merkato,2025-10-18 13:00:00,146.0,91.3,-54.7,0.0,0.0,0.0,0.0,0.0,1,Saturday
82357,Megenagna,2025-10-29 18:00:00,183.0,128.9,-54.1,3.5,0.0,0.0,0.0,0.0,0,Wednesday
79347,Bole,2025-10-19 00:00:00,130.0,76.2,-53.8,0.0,1.0,0.0,0.0,0.0,1,Sunday
81232,Kazanchis,2025-10-25 17:00:00,67.0,119.2,52.2,0.0,1.0,1.0,0.0,1.0,1,Saturday
79225,Kazanchis,2025-10-18 13:00:00,28.0,77.7,49.7,0.0,1.0,1.0,0.0,0.0,1,Saturday


Kazanchis  Sat 18 Oct 17:00  err  +71.2  events that day: Premier league match [confirmed]
Bole       Sat 18 Oct 22:00  err  -60.4  events that day: Trade exhibition [confirmed]
Bole       Sun 26 Oct 18:00  err  -58.4  events that day: Live concert [confirmed], Live concert [confirmed]
Megenagna  Tue 28 Oct 18:00  err  -58.1  events that day: —
Merkato    Sat 18 Oct 07:00  err  -57.3  events that day: —
Merkato    Sat 18 Oct 13:00  err  -54.7  events that day: —
Megenagna  Wed 29 Oct 18:00  err  -54.1  events that day: —
Bole       Sun 19 Oct 00:00  err  -53.8  events that day: Trade exhibition [confirmed]
Kazanchis  Sat 25 Oct 17:00  err  +52.2  events that day: Premier league match [confirmed]
Kazanchis  Sat 18 Oct 13:00  err  +49.7  events that day: Premier league match [confirmed]


**Interpretation:** (a) Absolute error is largest in the busiest zones (Merkato, Kazanchis, Bole, Megenagna: RMSE 11–13), but relative error is flat at 17–20% of mean in every zone, including the newest (Ayat 20%). By hour, absolute error peaks at the 07:00 and 17:00–18:00 rush (RMSE 12–15), relative error is highest at night (~30%, small counts). Weekends are slightly worse than weekdays (20.5% vs 17.4%), and the model has a small under-forecast bias (−1 trip). (b) The 10 largest errors fall in three groups: **Kazanchis football days (18 & 25 Oct)** over-predicted by 50–70 trips — the model applies the average match uplift, but these matches drew far less (possibly low-profile fixtures; attendance alone doesn't capture it); **Bole events (exhibition night 18 Oct, two concerts 26 Oct)** under-predicted by ~55–60 — overlapping/late-night events produce bigger surges than the average; and **unlisted spikes** — Megenagna evening rush on 28–29 Oct in rain (transport-hub surge larger than the average rain effect) and Merkato on Saturday 18 Oct morning/midday (no listed event; likely a market or religious-day crowd).

## D8 — Response to findings

In [41]:
# Change tested: from B3.4 / D7, demand jumps on the eve of holidays and the model has no feature for it,
# and events are under-predicted right after they end. Add (1) is_holiday_eve and (2) hours_since_event_end-style
# post-event intensity: attendance × post-window flag. Both are known at forecast time.
hol_days = set()
for _, e in events[(events.event_type == "public_holiday") & (events.status != "cancelled")].iterrows():
    hol_days.update(pd.date_range(e.start.normalize(), e.end.normalize()))
for df in (train, test):
    df["is_holiday_eve"] = (df.datetime.dt.normalize() + pd.Timedelta(days=1)).isin(hol_days).astype(int)
    df["post_event_attendance"] = df.event_post * df.event_attendance
NEW = ["is_holiday_eve", "post_event_attendance"]
tr_main, va_main = split(train, VAL_START)

d8 = []
for name, fs in [("final features", FEATS), ("+ holiday eve + post-event attendance", FEATS + NEW)]:
    r_main = rmse(va_main.trips, predict(fit(FINAL_KIND, tr_main, fs, BEST_PARAMS), va_main))
    r_roll = [rmse(v.trips, predict(fit(FINAL_KIND, t, fs, BEST_PARAMS), v)) for t, v in (split(train, c) for c in CUTS)]
    d8.append({"variant": name, "main_val_RMSE": r_main, "rolling_mean_RMSE": np.mean(r_roll)})
d8 = pd.DataFrame(d8); display(d8.round(3))
if d8.rolling_mean_RMSE.iloc[1] < d8.rolling_mean_RMSE.iloc[0]:
    FEATS = FEATS + NEW; EVENTS = EVENTS + NEW
    print("Kept the new features (better on the rolling folds).")
else:
    print("Not kept – no improvement on the rolling folds.")

,variant,main_val_RMSE,rolling_mean_RMSE
0,final features,9.058,9.670
1,+ holiday eve + post-event attendance,9.029,9.582


Kept the new features (better on the rolling folds).


**Interpretation:** Following B3.4 and D7, I added two forecast-time features: `is_holiday_eve` (the day before a public holiday) and `post_event_attendance` (attendance × post-event window, so big crowds leaving produce bigger surges). They improved the rolling-origin mean RMSE from 9.67 to **9.58** and the main fortnight from 9.06 to 9.03, so they were kept. The gain is modest: the largest D7 errors come from events that differ from their type's average and from unlisted events, which no calendar feature can fix.

## D10 — Optimisation round
Changes tested on all four rolling folds and kept only because each one lowered the mean rolling RMSE:
1. **Longer seasonal lags**: `lag_672h` and `lag_840h` (4 and 5 weeks back), plus their mean, median and spread across 2–5 weeks and the ratio `lag_336h / lag_504h`. All are at least 14 days old, so they are known on the forecast day.
2. **Recent profile `prof8`**: the mean trips for each zone × weekday × hour over the last 8 weeks of the *training slice*. It is recomputed inside every fold, so there is no leakage.
3. **Poisson objective**: trips are counts and the noise grows with the level.
4. **Smaller trees** (15 leaves, 1500 rounds) **averaged over 3 seeds**, which lowers variance.

In [42]:
def add_long_lags(train, test):
    full = pd.concat([train.assign(_t=1), test.assign(_t=0)], ignore_index=True)
    key = full.set_index(["zone", "datetime"]).index
    for h in (672, 840):
        src = full[["zone", "datetime", "trips"]].assign(datetime=lambda d: d.datetime + pd.Timedelta(hours=h))
        full[f"lag_{h}h"] = key.map(src.set_index(["zone", "datetime"]).trips.to_dict().get)
    L = ["lag_336h", "lag_504h", "lag_672h", "lag_840h"]
    full[L] = full[L].astype(float)
    full["wk_mean"], full["wk_med"], full["wk_std"] = full[L].mean(1), full[L].median(1), full[L].std(1)
    full["lag_ratio"] = full.lag_336h / full.lag_504h.clip(lower=1)
    full["hour_dow"] = full.hour * 7 + full.day_of_week
    return full[full._t == 1].drop(columns="_t").reset_index(drop=True), \
           full[full._t == 0].drop(columns="_t").reset_index(drop=True)

train, test = add_long_lags(train, test)
LONG = ["lag_672h", "lag_840h", "wk_mean", "wk_med", "wk_std", "lag_ratio", "hour_dow"]
LAGS = LAGS + LONG + ["prof8"]
V2_PARAMS = {**BEST_PARAMS, "objective": "poisson", "num_leaves": 15, "n_estimators": 1500}
V2_SEEDS = (1, 2, 3)

def _profile(tr):
    rec = tr[tr.datetime >= tr.datetime.max() - pd.Timedelta(weeks=8)]
    return rec.groupby(["zone", "day_of_week", "hour"]).trips.mean().rename("prof8")

_fit_old, _predict_old = fit, predict
def fit(kind, tr, feats, params=None):
    if kind != "lgb_v2":
        return _fit_old(kind, tr, feats, params)
    prof = _profile(tr)
    t2 = tr.join(prof, on=["zone", "day_of_week", "hour"])
    f = [c for c in feats if c != "prof8"] + ["prof8"]
    models = [lgb.LGBMRegressor(**{**BASE_LGB, **(params or V2_PARAMS), "random_state": s}).fit(X_of(t2, f), t2.trips)
              for s in V2_SEEDS]
    return {"kind": kind, "model": models, "features": f, "profile": prof}

def predict(fm, df):
    if fm["kind"] != "lgb_v2":
        return _predict_old(fm, df)
    d = df.drop(columns="prof8", errors="ignore").join(fm["profile"], on=["zone", "day_of_week", "hour"])
    p = np.mean([m.predict(X_of(d, fm["features"])) for m in fm["model"]], axis=0)
    return np.clip(p, 0, None)

tr_main, va_main = split(train, VAL_START)
V2_FEATS = FEATS + LONG + ["prof8"]
d10 = []
for name, kind, fs, prm in [("previous final (D6/D8)", FINAL_KIND, FEATS, BEST_PARAMS),
                            ("optimised v2", "lgb_v2", V2_FEATS, V2_PARAMS)]:
    r_main = rmse(va_main.trips, predict(fit(kind, tr_main, fs, prm), va_main))
    r_roll = [rmse(v.trips, predict(fit(kind, t, fs, prm), v)) for t, v in (split(train, c) for c in CUTS)]
    d10.append({"model": name, "main_val_RMSE": r_main, "rolling_mean_RMSE": np.mean(r_roll), "rolling_std": np.std(r_roll)})
d10 = pd.DataFrame(d10); display(d10.round(3))
if d10.rolling_mean_RMSE.iloc[1] < d10.rolling_mean_RMSE.iloc[0]:
    FINAL_KIND, FEATS, BEST_PARAMS = "lgb_v2", V2_FEATS, V2_PARAMS
    print("Optimised model adopted as final.")

,model,main_val_RMSE,rolling_mean_RMSE,rolling_std
0,previous final (D6/D8),9.029,9.582,0.618
1,optimised v2,8.471,9.212,0.706


Optimised model adopted as final.


**Interpretation:** The optimised model lowers main-validation RMSE from 9.03 to 8.47 (−6%) and mean rolling-origin RMSE from 9.58 to 9.21, beating the previous final on the folds, so it is adopted. Most of the gain comes from the 8-week zone×weekday×hour profile (a robust estimate of the 'normal' hour that adapts to recent growth) and the Poisson loss, which matches count data whose noise grows with demand. The extra 4–5-week lags smooth out one-off spikes in the 2–3-week lags. Seed averaging and smaller trees cut variance. Fold spread rises slightly (0.62→0.71), so the gain is real but not uniform across fortnights.

## D9 — Plain-language metric

In [43]:
final_val = fit(FINAL_KIND, tr_main, FEATS, BEST_PARAMS)
p = predict(final_val, va_main)
R, M, mean_d = rmse(va_main.trips, p), mae(va_main.trips, p), va_main.trips.mean()
TRIPS_PER_DRIVER = round((train.trips / train.active_drivers).median(), 2)
print(f"Validation RMSE {R:.2f} trips, MAE {M:.2f} trips per zone-hour; mean demand {mean_d:.1f} trips per zone-hour.")
print(f"MAE = {100*M/mean_d:.1f}% of mean demand, RMSE = {100*R/mean_d:.1f}%.")
print(f"At {TRIPS_PER_DRIVER} trips per driver-hour: typical miss ≈ {M/TRIPS_PER_DRIVER:.1f} drivers per zone-hour "
      f"(larger misses ≈ {R/TRIPS_PER_DRIVER:.1f} drivers).")

Validation RMSE 8.47 trips, MAE 5.85 trips per zone-hour; mean demand 33.4 trips per zone-hour.
MAE = 17.5% of mean demand, RMSE = 25.3%.
At 1.31 trips per driver-hour: typical miss ≈ 4.5 drivers per zone-hour (larger misses ≈ 6.5 drivers).


**Interpretation:** On the validation fortnight the forecast is off by **about 6 trips per zone-hour on a typical hour (MAE 6.1)** against an average of 33 trips — about 18% of demand. Big misses (RMSE 9.0, 27% of demand) happen mainly at rush hours and around events. At roughly 1.3 trips per driver-hour, that means the plan is usually off by **about 4–5 drivers per zone per hour**, and by ~7 drivers in the harder hours, so operations should keep a small buffer of 5–7 standby drivers per busy zone at peaks and on event days.

## Final model, submission and exports

In [44]:
# 1) Final model: retrain on ALL history with the chosen features/params
final = fit(FINAL_KIND, train, FEATS, BEST_PARAMS)
joblib.dump({**final, "params": BEST_PARAMS, "zones": ZONES, "trained_until": str(train.datetime.max()),
             "trips_per_driver": TRIPS_PER_DRIVER}, MOD / "final_model.joblib")

# 2) Submission in template order
tmpl = pd.read_csv(ROOT / "data" / "raw" / "submission_template.csv")
test["predicted_trips"] = predict(final, test)
sub = tmpl[["row_id"]].merge(test[["row_id", "predicted_trips"]], on="row_id", how="left")
assert len(sub) == 4032 and sub.row_id.is_unique and sub.row_id.tolist() == tmpl.row_id.tolist()
assert sub.predicted_trips.notna().all() and (sub.predicted_trips >= 0).all()
sub["predicted_trips"] = sub.predicted_trips.round(3)
sub.to_csv(SUB / "team_alpha_submission.csv", index=False)
print("Submission:", sub.shape, "| mean predicted", round(sub.predicted_trips.mean(), 2),
      "| mean Oct actual", round(train[train.datetime.dt.month == 10].trips.mean(), 2))

# 3) Exports for notebook 03 (fig10–fig12)
name_of = {"lgb": "LightGBM (raw target)", "lgb_norm": "LightGBM (level-normalised target)"}
mr = cmp_tbl[["model", "RMSE", "MAE"]].rename(columns={"RMSE": "rmse", "MAE": "mae"})
mr["rmse_std"] = np.nan
mr["is_baseline"] = mr.model.isin(base_tbl.model)
if FINAL_KIND == "lgb_v2":
    name_of["lgb_v2"] = "LightGBM optimised (Poisson + long lags + profile)"
    mr = pd.concat([mr, pd.DataFrame([{"model": name_of["lgb_v2"], "is_baseline": False}])], ignore_index=True)
mr.loc[mr.model == name_of[FINAL_KIND], "rmse"] = R          # tuned/final version
mr.loc[mr.model == name_of[FINAL_KIND], "mae"] = M
mr.loc[mr.model == name_of[FINAL_KIND], "rmse_std"] = d10.rolling_std.iloc[-1] if FINAL_KIND == "lgb_v2" else roll.final_RMSE.std()
mr.loc[mr.model == "Seasonal-naive (all history)", "rmse_std"] = roll.seasonal_RMSE.std()
mr.to_csv(REP / "model_results.csv", index=False)

va_out = va_main[["zone", "datetime", "trips"]].assign(predicted=p)
va_out.to_csv(REP / "val_predictions.csv", index=False)

# permutation importance on the validation fortnight (5 repeats)
rng = np.random.default_rng(SEED)
base_r = rmse(va_main.trips, p)
imp = []
va_imp = va_main.join(final_val["profile"], on=["zone", "day_of_week", "hour"]) if "profile" in final_val else va_main
_pred_imp = (lambda d: np.clip(np.mean([m.predict(X_of(d, final_val["features"])) for m in final_val["model"]], 0), 0, None)) \
            if FINAL_KIND == "lgb_v2" else (lambda d: predict(final_val, d))
for f in FEATS:
    inc = []
    for _ in range(5):
        v2 = va_imp.copy(); v2[f] = rng.permutation(v2[f].values)
        inc.append(rmse(v2.trips, _pred_imp(v2)) - base_r)
    grp = "weather" if f in WEATHER else "event" if f in EVENTS else "lag" if f in LAGS else "calendar"
    imp.append({"feature": f, "importance": np.mean(inc), "importance_std": np.std(inc), "group": grp})
imp = pd.DataFrame(imp).sort_values("importance", ascending=False)
imp.to_csv(REP / "feature_importance.csv", index=False)
display(imp.head(15).round(3))

Submission: (4032, 2) | mean predicted 32.81 | mean Oct actual 32.88


,feature,importance,importance_std,group
39,prof8,25.349,0.288,lag
34,wk_mean,0.505,0.081,lag
0,zone,0.488,0.045,calendar
15,rain_mm,0.378,0.017,weather
38,hour_dow,0.193,0.032,lag
2,day_of_week,0.150,0.011,calendar
13,roll_mean_168h_lag336,0.124,0.026,lag
35,wk_med,0.122,0.041,lag
1,hour,0.094,0.015,calendar
4,day_of_month,0.074,0.012,calendar


**Interpretation:** The final model is LightGBM retrained on all history (1 Jan – 31 Oct) with the D10 parameters and the 40 forecast-time features, saved with its feature list to `models/final_model.joblib`. The submission has all 4,032 rows in template order with no blank or negative values; mean predicted demand (32.8) is in line with the October average (32.9). Permutation importance is led by the 8-week zone × weekday × hour profile (`prof8`), followed by the multi-week lag mean and the zone; rain_mm is the strongest non-history input and the event features matter only in the few hours they touch, which matches the ablation.